# Low-Rank Adaptation
LoRA微调是目前在后训练阶段（post-training）最常用的微调方式之一，该方法通过引入低秩矩阵来近似模型每层的参数更新，从而减少了模型在适配下游任务时所需训练的参数量，并尽可能取得与全量微调相近的结果。

## LoRA原理
根据前面对模型结构的学习，我们知道，在大语言模型中包含有大量的线性变换层，线性变换层的参数矩阵的维度通常都很高。研究人员发现LLM在针对下游任务进行训练时，参数矩阵往往是过参数化（Over-parametrized）的，也就是针对特定的下游任务，并不是模型的全部参数都起重要，而是只有其中的一部分参数发挥了作用。而真正起作用的这部分参数，就是需要通过某种方法来近似表示。这就是“低秩”的作用。因此，LoRA提出在预训练模型的参数矩阵上添加低秩分解矩阵来近似每层的参数更新，从而减少适配下游任务所需要训练的参数。

具体原理如下：\
给定一个参数矩阵$\mathbf W$，其更新过程可以一般性地表达为以下形式：
$$\mathbf W= \mathbf W_0+ \Delta\mathbf W$$

其中，$\mathbf W_0$ 是原始参数矩阵，$\Delta\mathbf W$ 是更新的梯度矩阵。LoRA 的基本思想是冻结原 始矩阵 $\mathbf W_0 ∈ R^{H*H}$，通过低秩分解矩阵 $\mathbf A ∈ R^{H*H}$和  $\mathbf B ∈ R^{H*H}$ 来近似参数更新矩阵 $\Delta W=A\cdot B^T$，其中 $R << H$ 是减小后的秩。在微调期间，原始的矩阵参数 $W_0$不会被更新，低秩分解矩阵 $\mathbf A$ 和 $\mathbf B$则是可训练参数用于适配下游任务。

在前向传 播过程中，原始计算中间状态 $\mathbf h = \mathbf W_0 \cdot \mathbf x$ 的公式修改为:
$$\mathbf h = \mathbf W_0 \cdot x + \mathbf A \cdot \mathbf B^T \cdot x$$
在训练完成后，进一步将原始参数矩阵 $\mathbf W_0$ 和训练得到的权重  $\mathbf A$ 和 $\mathbf B$ 进行合并：
$$\mathbf W = \mathbf W_0 + \mathbf A \cdot \mathbf B^T$$
，得到更新后的参数矩阵。因此，LoRA 微调得到的模型在解码过 程中不会增加额外的开销。

![alt text](./_img/lora_img.png)

## LoRA作用在什么地方
注意：LoRA增加的旁路是在模型的线性变换层，主要是多头注意力的4个线性变换矩阵，即$W^Q, W^K, W^V, W^O$，以及全连接层。

回顾注意力机制：
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

其中$Q,K,V$的计算公式为：

$$Q=X_Q​W_Q​,K=X_K​W_K​,V=X_V​W_V​$$

### 线性层的LoRA实现

In [ ]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r):
        super(LoRALinear, self).__init__()
        self.in_features = in_features  # 对应 d
        self.out_features = out_features  # 对应 k
        self.r = r  # 低秩值

        # 原始权重矩阵，冻结
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.weight.requires_grad = False  # 冻结

        # LoRA 部分的参数，初始化 A 为全零，B 从均值为 0 的正态分布中采样
        self.A = nn.Parameter(torch.zeros(r, in_features))  # 形状为 (r, d)
        self.B = nn.Parameter(torch.empty(out_features, r))  # 形状为 (k, r)
        nn.init.normal_(self.B, mean=0.0, std=0.02)  # 初始化 B

        # 偏置项，可选
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, x):
        # 原始部分
        original_output = torch.nn.functional.linear(x, self.weight, self.bias)
        # LoRA 增量部分
        delta_W = torch.matmul(self.B, self.A)  # 形状为 (k, d)
        lora_output = torch.nn.functional.linear(x, delta_W)
        # 总输出
        return original_output + lora_output


### 注意力层的代码实现

In [ ]:
import torch
import torch.nn as nn

class LoRAAttention(nn.Module):
    def __init__(self, embed_dim, r):
        super(LoRAAttention, self).__init__()
        self.embed_dim = embed_dim  # 对应 d_model
        self.r = r  # 低秩值

        # 原始的 QKV 权重，冻结
        self.W_Q = nn.Linear(embed_dim, embed_dim)
        self.W_K = nn.Linear(embed_dim, embed_dim)
        self.W_V = nn.Linear(embed_dim, embed_dim)
        self.W_O = nn.Linear(embed_dim, embed_dim)

        for param in self.W_Q.parameters():
            param.requires_grad = False
        for param in self.W_K.parameters():
            param.requires_grad = False
        for param in self.W_V.parameters():
            param.requires_grad = False

        # LoRA 的 Q 部分
        self.A_Q = nn.Parameter(torch.zeros(r, embed_dim))
        self.B_Q = nn.Parameter(torch.empty(embed_dim, r))
        nn.init.normal_(self.B_Q, mean=0.0, std=0.02)

        # LoRA 的 K 部分
        self.A_K = nn.Parameter(torch.zeros(r, embed_dim))
        self.B_K = nn.Parameter(torch.empty(embed_dim, r))
        nn.init.normal_(self.B_K, mean=0.0, std=0.02)

        # LoRA 的 V 部分
        self.A_V = nn.Parameter(torch.zeros(r, embed_dim))
        self.B_V = nn.Parameter(torch.empty(embed_dim, r))
        nn.init.normal_(self.B_V, mean=0.0, std=0.02)

    def forward(self, query, key, value):
        """
        query, key, value: 形状为 (batch_size, seq_length, embed_dim)
        """
        # 计算原始的 Q、K、V
        Q = self.W_Q(query)  # (batch_size, seq_length, embed_dim)
        K = self.W_K(key)
        V = self.W_V(value)

        # 计算 LoRA 增量部分
        delta_Q = torch.matmul(query, self.A_Q.t())  # (batch_size, seq_length, r)
        delta_Q = torch.matmul(delta_Q, self.B_Q.t())  # (batch_size, seq_length, embed_dim)
        delta_K = torch.matmul(key, self.A_K.t())
        delta_K = torch.matmul(delta_K, self.B_K.t())
        delta_V = torch.matmul(value, self.A_V.t())
        delta_V = torch.matmul(delta_V, self.B_V.t())

        # 更新后的 Q、K、V
        Q = Q + delta_Q
        K = K + delta_K
        V = V + delta_V

        # 计算注意力得分
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.embed_dim ** 0.5)
        attn_weights = torch.nn.functional.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)

        # 输出层
        output = self.W_O(context)

        return output


## LoRA所需的显存估计
假设模型的原始参数量为$\mathbf P$，使用LoRA微调方式，假设需要微调的参数量为$\mathbf P_{LoRA}$，采用混合精度微调的方式，模型和梯度参数用`FP16`保存，优化器参数用`FP32`保存，那么在微调时，GPU显存中占用的内存主要分为：
1. 需要保存的模型参数量为：$2\mathbf P + 2\mathbf P_{LoRA}$
2. 保存梯度所需要的参数为：$2\mathbf P_{LoRA}$
3. 优化器需要保存一份模型参数，一阶动量矩参数和二阶动量矩参数，总参数量为：$4\mathbf P_{LoRA} + 4\mathbf P_{LoRA} + 4\mathbf P_{LoRA}$、

因此，LoRA微调总共需要的参数量为：$2\mathbf P + 16\mathbf P_{LoRA}$，其中%$P_{LoRA} = 4\cdot 2 \cdot L \cdot HR$

### 计算实例
以LLaMA 7B模型为例，模型层数$L=32$,中间状态维度为$H=4096$，秩$R=8$，则$P_{LoRA} = 4\cdot 2 \cdot 32 \cdot 4096 \dot 8 = 8388608$，模型总参数量为：$2\mathbf P + 16\mathbf P_{LoRA} = 14GB$

在全量微调的情况下，模型所需的内存为$16P=108GB$，由此可见，LoRA微调显著降低了模型微调所需的参数量。

## QLoRA
QLoRA将原始的参数矩阵量化为 4-bits，而低秩参数部分仍使用 16-bits进行训练。别的与LoRA相同。

# 代码实践

## 导包

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

In [ ]:
ds = Dataset.load_from_disk("/root/tuning/lesson01/data/alpaca_data_zh/")
ds